## Config — Lab 6 (Gold Layer)
Shared configuration for all Lab 6 notebooks. It does not process data —
it only loads `gold_config.yml` for the selected environment (`dev` / `prod`)
and builds full table names (`catalog.schema.table`)

In [0]:
dbutils.widgets.dropdown("env", "dev", ["dev", "prod"], "Environment")
env = dbutils.widgets.get("env")


In [0]:
import os
import yaml

config_path = os.path.join(os.getcwd(), "gold_config.yml")

with open(config_path, "r") as f:
    full_config = yaml.safe_load(f)

if env not in full_config:
    raise ValueError(f"Environment '{env}' not found in {config_path}")

config = full_config[env]

In [0]:
from pyspark.sql import functions as F

catalog = config["catalog"]
silver_schema = config["silver_schema"]
gold_schema = config["gold_schema"]

videos_silver = f"{catalog}.{silver_schema}.{config['source']['videos_silver']}"
gold = {t: f"{catalog}.{gold_schema}.{t}" for t in config["gold_tables"]}

period_start = config["trending_period"]["start"]
period_end = config["trending_period"]["end"]
reviewer_email = config["reviewer_email"]


def scoped_silver():
    """Silver rows within the business period, with a typed trending date."""
    return (
        spark.table(videos_silver)
        .withColumn("trending_dt", F.to_date("trending_date", "yy.dd.MM"))
        .filter(F.col("trending_dt").between(period_start, period_end))
    )